# 05 - Federated Learning Stage

Run measured non-IID FedAvg, FedProx, CFL-style clustered aggregation, and RareGuardFed training. Keep the default CPU settings for smoke tests, then increase rounds/epochs after the pipeline is stable.


In [5]:
from pathlib import Path
import importlib.util
import os, sys

LOCAL_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION')

def is_project_root(path: Path) -> bool:
    try:
        return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()
    except OSError:
        return False

def find_project_root() -> Path | None:
    if Path('/content').is_dir():
        os.chdir('/content')
        if not is_project_root(DRIVE_PROJECT_ROOT):
            from google.colab import drive
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            drive.mount('/content/drive', force_remount=True)
        return DRIVE_PROJECT_ROOT if is_project_root(DRIVE_PROJECT_ROOT) else None
    return LOCAL_PROJECT_ROOT if is_project_root(LOCAL_PROJECT_ROOT) else None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. '
        'If you are using Colab connector, Colab cannot read /Volumes/LocalDrive1 directly. '
        'Copy/sync the IMPLEMENTATION folder to Google Drive at '
        '/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION, '
        'or run this notebook with a local VS Code Python kernel on your Mac.'
    )
os.chdir(PROJECT_ROOT)
print('Drive project root:', PROJECT_ROOT)
for name in ['logs', 'results', 'processed', 'features', 'models', 'figures', 'reports']:
    (PROJECT_ROOT / 'artifacts' / name).mkdir(parents=True, exist_ok=True)
print('Drive artifact folders ready:', PROJECT_ROOT / 'artifacts')


Drive project root: /Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Drive artifact folders ready: /Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts


In [2]:
DATASET_KEY = 'all'
DEVICE = 'cpu'
runner_path = PROJECT_ROOT / 'scripts/run_federated_resilient.py'
spec = importlib.util.spec_from_file_location('rareguard_resilient_runner', runner_path)
resilient_runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(resilient_runner)
RUNTIME_ROOT = resilient_runner.run_resilient_federated(
    drive_root=PROJECT_ROOT,
    dataset=DATASET_KEY,
    device=DEVICE,
)
print('Stable runtime root:', RUNTIME_ROOT)


2026-10-23 01:18:55,982 | INFO | rareguard | Federated run device=cpu resume=True completed_pairs=6
2026-10-23 01:18:56,454 | INFO | rareguard | Skipping completed cse_cic_ids2018 FedAvg
2026-10-23 01:18:56,455 | INFO | rareguard | Skipping completed cse_cic_ids2018 FedProx
2026-10-23 01:18:56,455 | INFO | rareguard | Skipping completed cse_cic_ids2018 CFL
2026-10-23 01:18:56,455 | INFO | rareguard | Skipping completed cse_cic_ids2018 RareGuardFed
2026-10-23 01:18:56,727 | INFO | rareguard | Skipping completed cic_bccc_nrc_tabulariotattack_2024 FedAvg
2026-10-23 01:18:56,727 | INFO | rareguard | Skipping completed cic_bccc_nrc_tabulariotattack_2024 FedProx
2026-10-23 01:18:56,728 | INFO | rareguard | Starting cic_bccc_nrc_tabulariotattack_2024 CFL
2026-10-23 01:19:45,159 | INFO | rareguard | cic_bccc_nrc_tabulariotattack_2024 CFL f1_macro=0.9909
2026-10-23 01:19:45,257 | INFO | rareguard | Starting cic_bccc_nrc_tabulariotattack_2024 RareGuardFed
2026-10-23 01:19:47,884 | INFO | raregua

In [3]:
import pandas as pd

summary_path = RUNTIME_ROOT / 'artifacts/results/federated_summary.csv'
rounds_path = RUNTIME_ROOT / 'artifacts/results/federated_round_metrics.csv'
if not summary_path.exists() or not rounds_path.exists():
    raise RuntimeError('Federated outputs are missing. Run the training cell successfully before displaying results.')
fl = pd.read_csv(summary_path)
round_metrics = pd.read_csv(rounds_path)
display(fl)
print('Summary rows:', len(fl), '| Round rows:', len(round_metrics))


,dataset,method,clients,rounds,non_iid_alpha,dropout_rate,avg_participating_clients,communication_units,model_family,evidence,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,cse_cic_ids2018,FedAvg,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.749200,0.249308,0.358860,0.291393,0.654282
1,cse_cic_ids2018,FedProx,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.740400,0.267190,0.350347,0.298772,0.645279
2,cse_cic_ids2018,CFL,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.747800,0.248819,0.357502,0.290711,0.652990
3,cse_cic_ids2018,RareGuardFed,5,5,0.5,0.1,4.6,23,RareGuard,measured_federated_training,0.987400,0.787133,0.781788,0.784233,0.984039
4,cic_bccc_nrc_tabulariotattack_2024,FedAvg,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.997106,0.995642,0.986218,0.990874,0.997092
5,cic_bccc_nrc_tabulariotattack_2024,FedProx,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.995527,0.989283,0.982643,0.985935,0.995512
6,cic_bccc_nrc_tabulariotattack_2024,CFL,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.997106,0.995642,0.986218,0.990874,0.997092
7,cic_bccc_nrc_tabulariotattack_2024,RareGuardFed,5,5,0.5,0.1,4.6,23,RareGuard,measured_federated_training,0.999079,0.999496,0.994753,0.997110,0.999077
8,cic_iiot_2025_datasense,FedAvg,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.593100,0.053282,0.115074,0.065538,0.530398
9,cic_iiot_2025_datasense,FedProx,5,5,0.5,0.1,4.6,23,FalconID,measured_federated_training,0.581100,0.049355,0.100091,0.059399,0.525019


Summary rows: 16 | Round rows: 80


In [4]:
# Safe to rerun: copies the local federated checkpoints to Google Drive.
resilient_runner.sync_federated_artifacts(RUNTIME_ROOT, PROJECT_ROOT, remount=True)


Federated checkpoints already reside in the canonical project
